# Joint Effect V2 — Temperature × Prompt Specificity

**What this tests:** Does increasing temperature amplify stereotype generation more under implicit prompts than under explicit prompts? If yes, that is the joint effect.

**Key improvements over V1:**
- 10 temperature values instead of 4
- Continuous outcome: female marker proportion (not binary)
- Proper interaction test: regression with `temperature × tier`
- 8-bit quantization
- Fixed CEO prompt (now counter-stereotypical)

**Steps:** Run cells 1 → 2 → 3 → 4 → 5 in order. Cell 3 takes ~90 min.

In [ ]:
# ── CELL 1: Mount Drive and set up folder ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Change this path if you uploaded the folder somewhere else
FOLDER = '/content/drive/MyDrive/joint_v2_colab'

os.chdir(FOLDER)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── CELL 2: Install dependencies and log into Hugging Face ─────────────────
!pip install transformers accelerate bitsandbytes scipy statsmodels matplotlib pandas -q

from huggingface_hub import login
login()  # paste your HuggingFace token when prompted (needs access to Llama)

In [ ]:
# ── CELL 3: Run generation ──────────────────────────────────────────────────
# Generates 1,350 stories (10 temps x 5 seeds x 3 tiers x 9 roles x 1 model)
# Saves to outputs/generations_v2.csv incrementally (inside this folder).
# Safe to interrupt and re-run — it resumes from where it stopped.
# Estimated time: ~3 hours on a T4 GPU.

!python run_generation_v2.py

In [ ]:
# ── CELL 4: Run analysis ────────────────────────────────────────────────────
# Annotates gender markers, runs interaction regressions, saves figures.

!python analyze_joint_v2.py

In [ ]:
# ── CELL 5: Show figures inline ─────────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

for fig_path in sorted(Path('outputs/figures').glob('*.png')):
    print(f'\n--- {fig_path.name} ---')
    display(Image(str(fig_path)))

In [ ]:
# ── CELL 6: Download all results as a zip ───────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('joint_v2_results', 'zip', 'outputs')
files.download('joint_v2_results.zip')

print('Download started. The zip contains:')
print('  generations_v2.csv  — all generated stories')
print('  annotations_v2.csv  — gender markers per story')
print('  figures/            — interaction plots')